# SQL reference 4: joins

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-04-joins.ipynb)

Every join type and its row counts, `ON` versus `USING`, semi and anti joins, filters in `ON` versus `WHERE`, range and self joins, `LATERAL`, fan-out and how to avoid double counting, and join hints. Practise it in [SQL practice 4: joins](sql-practice-04-joins.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Join types](#join-types) | [`INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`](#inner-left-right-full-and-cross-join), [`ON`, `USING` and `NATURAL`](#on-using-and-natural), [Semi and anti joins](#semi-and-anti-joins) |
| [Join conditions](#join-conditions) | [Filters in `ON` versus `WHERE`](#filters-in-on-versus-where), [Non-equi and range joins](#non-equi-and-range-joins), [Self joins](#self-joins), [`LATERAL` joins](#lateral-joins) |
| [Correctness and performance](#correctness-and-performance) | [Fan-out: why totals double](#fan-out-why-totals-double), [Join hints and strategies](#join-hints-and-strategies) |

## Join types

### `INNER`, `LEFT`, `RIGHT`, `FULL` and `CROSS JOIN`

```sql
FROM a [INNER | LEFT [OUTER] | RIGHT [OUTER] | FULL [OUTER] | CROSS] JOIN b ON condition | USING (cols)
```

| join | keeps | unmatched columns |
| --- | --- | --- |
| `INNER JOIN` | pairs that match | (none kept) |
| `LEFT JOIN` | every `a` row, plus matching `b` rows | `b` columns NULL |
| `RIGHT JOIN` | every `b` row | `a` columns NULL (rewrite as a `LEFT JOIN` for readability) |
| `FULL JOIN` | every row of both | NULL on the missing side |
| `CROSS JOIN` | every combination (rows of `a` × rows of `b`) | no condition |

**Notes:** a row on one side that matches *n* rows on the other appears *n* times; only a join on a unique key of
the other table keeps the row count. NULL keys never match with `=`.

In [ ]:
show("""
SELECT c.customer_id, c.name, o.order_id, o.amount
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE c.customer_id IN (1, 6, 7)
ORDER BY c.customer_id, o.order_id
""")
show("""
SELECT (SELECT count(*) FROM customers) AS customers, (SELECT count(*) FROM orders) AS orders,
       (SELECT count(*) FROM customers c JOIN orders o USING (customer_id)) AS inner_rows,
       (SELECT count(*) FROM customers c LEFT JOIN orders o USING (customer_id)) AS left_rows,
       (SELECT count(*) FROM customers CROSS JOIN orders) AS cross_rows
""")

### `ON`, `USING` and `NATURAL`

| form | behaviour |
| --- | --- |
| `ON a.k = b.k AND ...` | any boolean condition; both `k` columns stay in the output |
| `USING (k1, k2)` | equality on same-named columns, output has **one** `k` column (coalesced in a `FULL JOIN`) |
| `NATURAL JOIN` | `USING` on every same-named column: fragile, a new column silently changes the join; avoid |

**Notes:** with `ON`, selecting an unqualified column that exists on both sides raises an ambiguity error: alias your
tables and qualify columns.

In [ ]:
show("SELECT customer_id, name, order_id FROM customers JOIN orders USING (customer_id) WHERE order_id <= 3 ORDER BY order_id")
with expected_error():
    show("SELECT customer_id FROM customers c JOIN orders o ON o.customer_id = c.customer_id")

### Semi and anti joins

| join | keeps | equivalent |
| --- | --- | --- |
| `a LEFT SEMI JOIN b ON ...` | `a` rows with **at least one** match; `a` columns only, never duplicated | `WHERE EXISTS (SELECT 1 FROM b WHERE ...)` |
| `a LEFT ANTI JOIN b ON ...` | `a` rows with **no** match | `WHERE NOT EXISTS (...)` |

**Notes:** `SEMI`/`ANTI` are Spark (and DuckDB) syntax; `EXISTS`/`NOT EXISTS` are portable and Spark plans them as
the same joins. Prefer them over `IN`/`NOT IN` (NULL trap) and over an inner join plus `DISTINCT` (duplicates, extra
shuffle).

In [ ]:
show("SELECT customer_id, name FROM customers c LEFT SEMI JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'paid' ORDER BY 1")
show("SELECT customer_id, name FROM customers c LEFT ANTI JOIN orders o USING (customer_id) ORDER BY 1")

## Join conditions

### Filters in `ON` versus `WHERE`

For an **outer** join the placement of a filter changes the result:

| filter on the optional side | effect |
| --- | --- |
| in `ON` | decides which rows **match**; every preserved row is still returned, with NULLs when nothing matches |
| in `WHERE` | runs **after** the join and removes rows, including those NULL-extended rows: the left join becomes an inner join |

For an inner join both placements are equivalent.

In [ ]:
show("""
SELECT c.customer_id, count(o.order_id) AS paid_orders
FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id AND o.status = 'paid'
GROUP BY c.customer_id ORDER BY 1
""")
show("""
SELECT c.customer_id, count(o.order_id) AS paid_orders
FROM customers c LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.status = 'paid'                -- customers without paid orders vanish
GROUP BY c.customer_id ORDER BY 1
""")

### Non-equi and range joins

Join conditions can be any predicate: ranges (`BETWEEN`), inequalities, or validity intervals.

| pattern | condition |
| --- | --- |
| bucket lookup | `a.amount >= t.min_amount AND a.amount < t.max_amount` |
| value valid at a time (SCD type 2) | `o.order_ts >= p.valid_from AND (o.order_ts < p.valid_to OR p.valid_to IS NULL)` |
| pairs within a group | `a.k = b.k AND a.id < b.id` (each unordered pair once) |

**Notes:** without an equality, Spark cannot hash-partition and falls back to a broadcast nested loop join, which is
fine when one side is small and very slow when neither is. Add an equality key (a date, a bucket) where you can.

In [ ]:
show("""
WITH tiers AS (SELECT * FROM VALUES ('bronze', 0, 100), ('silver', 100, 500), ('gold', 500, 1000000) AS t(tier, lo, hi))
SELECT o.order_id, o.amount, t.tier
FROM orders o JOIN tiers t ON o.amount >= t.lo AND o.amount < t.hi
WHERE o.order_id <= 6 ORDER BY o.order_id
""")

### Self joins

A table joined to itself, with two aliases, compares rows of the same table: an employee and their manager, an event
and the next event, two products in the same order.

In [ ]:
show("""
SELECT e.name AS employee, m.name AS manager, e.salary, m.salary AS manager_salary
FROM employees e LEFT JOIN employees m ON m.emp_id = e.manager_id
ORDER BY e.emp_id
""")

### `LATERAL` joins

`LATERAL` lets a subquery in `FROM` reference columns of tables to its left, running once per left row (a
"for each" join). It expresses "the top *n* related rows per row" directly.

**Notes:** Spark, Postgres, Snowflake and DuckDB support `LATERAL`; SQL Server spells it `CROSS APPLY` / `OUTER APPLY`.
Use `LEFT JOIN LATERAL (...) ON TRUE` to keep rows with no match. Window functions are the usual alternative.

In [ ]:
show("""
SELECT c.customer_id, c.name, top.order_id, top.amount
FROM customers c
LEFT JOIN LATERAL (
    SELECT order_id, amount FROM orders o WHERE o.customer_id = c.customer_id ORDER BY amount DESC LIMIT 1
) top ON TRUE
ORDER BY c.customer_id
""")

## Correctness and performance

### Fan-out: why totals double

Joining a table to another at a finer grain repeats each row once per match. Summing a column of the coarser table
after such a join counts it several times. Order 1 has two item rows, so its amount is counted twice below.

**Fix:** aggregate each table to the grain of the join **before** joining, or aggregate only columns of the finest
table. A quick check: compare `count(*)` before and after the join, and compare against `count(DISTINCT key)`.

In [ ]:
show("""
SELECT o.customer_id, sum(o.amount) AS wrong_revenue, count(*) AS rows
FROM orders o JOIN order_items i USING (order_id)
WHERE o.customer_id = 1 GROUP BY o.customer_id
""")
show("""
WITH items AS (SELECT order_id, sum(qty) AS units FROM order_items GROUP BY order_id)
SELECT o.customer_id, sum(o.amount) AS revenue, sum(i.units) AS units
FROM orders o JOIN items i USING (order_id)
WHERE o.customer_id = 1 GROUP BY o.customer_id
""")

### Join hints and strategies

Spark picks a physical join from table sizes and statistics; hints override it:

| hint | strategy | when |
| --- | --- | --- |
| `/*+ BROADCAST(t) */` | broadcast hash join: copy `t` to every executor, no shuffle | one side is small (default threshold 10 MB) |
| `/*+ MERGE(t) */` | sort-merge join: shuffle and sort both sides | two large tables |
| `/*+ SHUFFLE_HASH(t) */` | shuffle hash join | large tables where one side's partitions fit in memory |

**Notes:** adaptive query execution can switch strategies at run time. Read the plan to see what was chosen
([query plans](sql-reference-12-query-plans-and-performance.ipynb)).

In [ ]:
text = plan("SELECT /*+ BROADCAST(c) */ o.order_id, c.name FROM orders o JOIN customers c USING (customer_id)")
print([line.strip() for line in text.splitlines() if "Join" in line][:2])

**Next:** practise these entries in [SQL practice 4: joins](sql-practice-04-joins.ipynb).